# Notes

---

### Pydantic

#### field_validator()

- field_validator is custom validation logic that runs on a single field
    - **@field_validator('FIELD')** is written above a class method
- field validation occurs after pydantic parses tokens and type checks it
    - it operates on specific fields that are declared
- it is one final step added before the field it truly accepted
- multiple @field_validator decorators can exist within one class 

In [ ]:
@field_validator("valve_callout") # type: ignore
def validate_and_parse_valve_callout(cls, v, info):
    try:
        parsed = parse_valve_callout(v) #type: ignore
    except ValueError as e:
        raise ValueError(f"Invalid valve callout: {e}")

    # Attach parsed valves to the model instance
    info.data["_parsed_valves"] = parsed
    return v

The above example shows a field validator being placed on the "valve_callout" field

---

### property decorator

- the property decorator is used to create managed attributes in classes
- it acts like and attribute of the class 
- commonly used for getter, setter, and deleter methods 

In [ ]:
class Person: 
    def __init__(self, name):
        self._name = name  # underscore to indicate "private"

    def get_name(self):
        return self._name

    def set_name(self, value):
        if not value:
            raise ValueError("Name cannot be empty")
        self._name = value

# p = Person("Alice")
# print(p.get_name())
# p.set_name("Bob")

In [ ]:
class Person:
    def __init__(self, name):
        self._name = name

    @property
    def name(self):
        """Getter for name"""
        return self._name

    @name.setter
    def name(self, value):
        """Setter for name"""
        if not value:
            raise ValueError("Name cannot be empty")
        self._name = value

    @name.deleter
    def name(self):
        """Deleter for name"""
        print("Deleting name...")
        del self._name

# p = Person("Alice")

# print(p.name)   # Calls the getter
# p.name = "Bob"  # Calls the setter
# del p.name      # Calls the deleter

### Pydantic Syntax

In [ ]:
class Mounting_And_Nameplate_Model(BaseModel):
    symbol: str # --> this field is required
    catalog_value: str | None = None # --> This field may be a string OR None, and its default value is None.

    model_config = {"extra": "ignore"} # any fields passed that are not declared in model will be dropped and validation will continue

### model_validator

- only one model_validator(mode="before") and model_validator(mode="after") section per class
- multiple functions can be defined in each section

Overall:
- model validator is a hook that lets you run logic on the entire model, not just one field
    - you can inspect mulitple fields together
    - enforce cross field rules
    - create or transform fields based on other fields
    - run post processing

- **model_validator("mode=before")**
    - receive: raw dict of the input data (exactly as passed by user, etc)
    - you can:
        - Add new fields
        - Remove fields
        - Rename fields
        - Combine or split fields
        - Preprocess the entire payload
        - Fix malformed input before validation happens
    - when to use:
        - When the incoming data is messy
        - When you need to restructure the input
        - When you want to derive a field before validation
        - When you want to enforce rules on raw input

- **model_validator("mode=after")**
    - receive: a fully typed model instance (self); all fields are validated, converted, and available
    - you can:
        - Compute derived fields
        - Enforce cross‑field constraints
        - Raise errors based on relationships between fields
        - Normalize or post‑process values
        - Populate additional attributes
    - when to use:
        - When you need to work with validated data
        - When you want to compute new attributes
        - When you want to enforce rules between fields
        - When you want to mutate the model after creation

### Pydantic Validation Order

**Validation Order (Simplified)**

<u>First --> Field Validators (all modes)</u>

These run first, in this order:

- @field_validator(..., mode="before")

- Built‑in type validation

- @field_validator(..., mode="after")

- This happens for each field individually, before the model is assembled.

--- 

<u>Second --> Model Validators</u>

After all fields are validated:

- @model_validator(mode="before")

- Model is constructed

- @model_validator(mode="after")